# Lab 3: Click-Driven Point-in-Polygon Classification

This lab uses map clicks to answer a spatial question:

**Which region contains the clicked point?**

Students will:

- display polygon regions as GeoJSON
- click anywhere on the map
- test whether the clicked point falls inside a polygon
- report the matching region in a widget

## Learning Goals

- display polygon data with `GeoJSON`
- convert a click into a point
- classify a point using a point-in-polygon function
- connect spatial analysis to map interaction

In [ ]:
from ipyleaflet import Map, Marker, GeoJSON, LayersControl, WidgetControl, basemaps, basemap_to_tiles
import ipywidgets as widgets
from IPython.display import display

## Sample GeoJSON

These are simple rectangular practice regions so the lab works immediately.
You can replace them later with states, counties, campus zones, or other course data.

In [ ]:
regions_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Western Zone", "color": "steelblue"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-100.5, 33.0],
                    [-97.5, 33.0],
                    [-97.5, 35.5],
                    [-100.5, 35.5],
                    [-100.5, 33.0]
                ]]
            }
        },
        {
            "type": "Feature",
            "properties": {"name": "Eastern Zone", "color": "darkorange"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-97.5, 33.0],
                    [-94.5, 33.0],
                    [-94.5, 35.5],
                    [-97.5, 35.5],
                    [-97.5, 33.0]
                ]]
            }
        }
    ]
}

## Point-in-Polygon Helper

This version uses the classic ray-casting method.
The polygon coordinates are GeoJSON-style `(lon, lat)` pairs.

In [ ]:
def ray_casting(point, polygon):
    lon, lat = point
    inside = False
    n = len(polygon)

    for i in range(n):
        lon1, lat1 = polygon[i]
        lon2, lat2 = polygon[(i + 1) % n]

        intersects = ((lat1 > lat) != (lat2 > lat))
        if intersects:
            boundary_lon = lon1 + (lat - lat1) * (lon2 - lon1) / (lat2 - lat1)
            if lon < boundary_lon:
                inside = not inside

    return inside


def containing_feature(lon, lat, feature_collection):
    for feature in feature_collection["features"]:
        polygon = feature["geometry"]["coordinates"][0]
        if ray_casting((lon, lat), polygon):
            return feature
    return None

In [ ]:
def style_callback(feature):
    color = feature["properties"].get("color", "gray")
    return {
        "color": color,
        "fillColor": color,
        "fillOpacity": 0.25,
        "weight": 2,
    }


geojson_layer = GeoJSON(data=regions_geojson, style=style_callback)

status = widgets.HTML(value="<b>Status:</b> Click inside or outside a region.")
log = widgets.Output()
panel = widgets.VBox([status, log])

m = Map(center=(34.25, -97.5), zoom=7)
m.add(basemap_to_tiles(basemaps.OpenStreetMap.Mapnik))
m.add(geojson_layer)
m.add(LayersControl())
m.add(WidgetControl(widget=panel, position="topright"))

click_marker = None

display(m)

## Click to Classify

The click callback turns map interaction into a spatial query.

In [ ]:
def handle_map_click(**kwargs):
    global click_marker

    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    if click_marker is not None:
        m.remove(click_marker)

    click_marker = Marker(location=(lat, lon))
    m.add(click_marker)

    feature = containing_feature(lon, lat, regions_geojson)

    if feature is None:
        region_name = "No region matched"
    else:
        region_name = feature["properties"]["name"]

    status.value = (
        f"<b>Status:</b> Click at ({lat:.4f}, {lon:.4f}) -> {region_name}"
    )

    with log:
        print(f"Clicked ({lat:.4f}, {lon:.4f}) -> {region_name}")


m.on_interaction(handle_map_click)

## Questions

1. Why do GeoJSON polygon coordinates use `(lon, lat)` instead of `(lat, lon)`?
2. Why does the click event return `(lat, lon)` while the polygon uses `(lon, lat)`?
3. What would need to change if a feature used multiple polygons instead of one?

## Mini Challenges

1. Replace the sample rectangles with course GeoJSON data.
2. Add different marker colors for inside vs. outside clicks.
3. Show the clicked region color or another property in the status panel.
4. Extend the helper to return all containing features instead of only the first one.